# Aditya-L1 in-situ payloads — common hands-on overview

## MAG + ASPEX-SWIS + ASPEX-STEPS 

---

### Purpose of this notebook

This is the **shared starting point for both projects**. By the end of it everyone — whichever
project group they join afterwards — will have:

1. built a small set of time helpers (`t_unix`, `t_str`, `t_dt`) instead of importing a
   mission-analysis framework,
2. read and quality-checked **all three in-situ payloads** of Aditya-L1,
3. computed the standard **solar-wind plasma parameters** from the raw moments,
4. put instruments of three different cadences on one **stack plot**,
5. read that stack plot as a single physical story.

Then the groups split:

* **Project 1** — identify the CME substructures: shock, sheath, magnetic cloud.
* **Project 2** — quantify the geoeffectiveness: propagate to the bow shock, compare with
  SYM-H, identify the storm phases.

### The three in-situ payloads

Aditya-L1 orbits the Sun–Earth **L1** point, ~1.5 million km sunward of Earth, and watches the
solar wind ~40 minutes before it reaches us. Three of its payloads measure the plasma *in situ*:

| Payload | Measures | Energy / quantity | Cadence (L2) |
|---|---|---|---|
| **MAG** | magnetic field | vector $\mathbf{B}$, GSE and GSM | 10 s |
| **ASPEX-SWIS** | solar-wind ions | bulk moments: $n$, $V$, $w$ (protons, alphas) | 5 s |
| **ASPEX-STEPS** | energetic ions | 0.25 – >20 MeV, 6 look directions | 10 min |

## 1. Setup —

This notebook deliberately uses **no mission-analysis framework**: the plotting is plain
`matplotlib`, and the only other libraries are `numpy` and the two file readers that the data
formats require (`netCDF4` for MAG's `.nc` files, `cdflib` for SWIS's `.cdf` files — no plotting
library can read those formats on its own).

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm
from datetime import datetime, timezone, timedelta

from netCDF4 import Dataset      # Aditya-L1 MAG L2 files are NetCDF-4
import cdflib                    # Aditya-L1 ASPEX-SWIS L2 files are CDF

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = False

# NaN-aware numpy functions warn about all-NaN slices; gaps are handled deliberately
# everywhere below, so silence the noise.
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

# --- Physical constants (SI unless stated) -------------------------------------
MP   = 1.6726219e-27     # proton mass            [kg]
KB   = 1.380649e-23      # Boltzmann constant     [J/K]
MU0  = 4e-7 * np.pi      # vacuum permeability    [H/m]
RE   = 6371.2            # Earth radius           [km]

BXYZ = ["blue", "green", "red"]     # conventional colours for Bx, By, Bz

print("Setup complete.")

### Time handling

All Aditya-L1 L2 products and all the indices we use can be reduced to **UNIX seconds**
(seconds since 1970-01-01 00:00:00 UTC), which is just a float — easy to sort, subtract,
average and bin. We convert to a `datetime` only at the last moment, for the plot axis.

Note `timezone.utc` everywhere: naive `datetime.utcfromtimestamp` is deprecated and, worse,
silently produces timezone-unaware objects that matplotlib may then interpret as local time.

In [ ]:
EPOCH = datetime(1970, 1, 1, tzinfo=timezone.utc)


def t_unix(s):
    """'2024-10-10' or '2024-10-10 14:45:30' (UTC) -> UNIX seconds."""
    s = s.strip()
    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d"):
        try:
            return (datetime.strptime(s, fmt).replace(tzinfo=timezone.utc)
                    - EPOCH).total_seconds()
        except ValueError:
            continue
    raise ValueError(f"unrecognised time string: {s!r}")


def t_str(u, fmt="%Y-%m-%d %H:%M:%S"):
    """UNIX seconds -> UTC string."""
    if not np.isfinite(u):
        return "n/a"
    return datetime.fromtimestamp(float(u), tz=timezone.utc).strftime(fmt)


def t_dt(u):
    """UNIX seconds (scalar or array) -> timezone-aware datetime(s) for matplotlib."""
    u = np.asarray(u, dtype="float64")
    if u.ndim == 0:
        return datetime.fromtimestamp(float(u), tz=timezone.utc)
    return np.array([datetime.fromtimestamp(float(s), tz=timezone.utc) for s in u])


# quick self-test
assert t_str(t_unix("2024-10-10 14:45:30")) == "2024-10-10 14:45:30"
print("time helpers OK  ->", t_str(t_unix("2024-10-10")), "=",
      t_unix("2024-10-10"), "UNIX s")

## 2. Reading the Aditya-L1 MAG Level-2 files

MAG L2 gives the interplanetary magnetic field at **10 s** cadence in GSE and GSM, plus a
per-sample quality flag (`1` = good, `0` = bad/missing) and the fill value `-9999.0`.

In [ ]:
import xarray as xr
import pandas as pd

MAG_DIR = os.path.join("Data", "mag")

# --- which dates are available on disk? ----------------------------------------
all_files = sorted(glob.glob(os.path.join(MAG_DIR, "L2_AL1_MAG_*.nc")))
available = [os.path.basename(f).split("_")[3] for f in all_files]   # 'L2_AL1_MAG_20241010_V00.nc' -> '20241010'
print("dates available:", available)

# --- CHOOSE THE DATES YOU WANT (format YYYYMMDD) -------------------------------
DATES = ["20241010", "20241011"]          # e.g. just one day: DATES = ["20241011"]

# --- pick out the matching files -----------------------------------------------
files = []
for d in DATES:
    hits = sorted(glob.glob(os.path.join(MAG_DIR, f"L2_AL1_MAG_{d}_*.nc")))
    if not hits:
        print(f"  !! no MAG file for {d}")
    files += hits

assert files, "none of the requested dates were found"
print("\nfiles selected:")
for f in files:
    print("  ", os.path.basename(f))

# --- open them all (decode_times=False: 'time' is plain UNIX seconds) -----------
dsets = [xr.open_dataset(f, decode_times=False) for f in files]

dsets[0]        # summary of the first file

## 3. Printing the Aditya-L1 MAG Level-2 file properties

In [ ]:
print("dimensions:", dict(dsets[0].sizes))
print()
print(f"{'variable':28s} {'shape':>10s}  units")
print("-" * 55)
for name in dsets[0].variables:
    v = dsets[0][name]
    print(f"{name:28s} {str(v.shape):>10s}  {v.attrs.get('units', '-')}")

## 4. Constructing a Pandas Dataframe

In [ ]:
def to_frame(ds):
    """One xarray dataset -> one pandas DataFrame, indexed by UTC time."""
    data = {name: ds[name].values for name in ds.variables if name != "time"}
    time = pd.to_datetime(ds["time"].values, unit="s")
    return pd.DataFrame(data, index=time)

df = pd.concat([to_frame(ds) for ds in dsets])   # all chosen days, one table
df = df.replace(-9999.0, np.nan)
df = df.sort_index()

print(df.index[0], "->", df.index[-1], " rows:", len(df))
df.head()

### Quality control of MAG

Four independent checks: fill values, non-finite values, the producer's quality flag, and the
time axis itself (monotonic? duplicated? gaps?).

In [ ]:
B_COLS = ["Bx_gse", "By_gse", "Bz_gse", "Bx_gsm", "By_gsm", "Bz_gsm"]

# count / min / max for the six field components
print(df[B_COLS].describe().T[["count", "min", "max"]].round(2))

# how many samples are missing (we already turned -9999 into NaN)
print("\nmissing values:")
print(df[B_COLS].isna().sum())

# the producer's own quality flag: 1 = good
print("\nquality flag:")
print(df["Quality_flag_10s_data"].value_counts())

# is the time axis clean? (should be one constant 10 s step, no gaps)
dt = df.index.to_series().diff().dt.total_seconds()
print("\ntime steps found (s):", sorted(dt.dropna().unique()))
print("gaps longer than 15 s:", int((dt > 15).sum()))

## 5. Drop the samples the instrument team flagged as bad

In [ ]:
bad = df["Quality_flag_10s_data"] != 1
df.loc[bad, B_COLS] = np.nan
print("samples removed by the quality flag:", int(bad.sum()))

# field magnitude (same number in GSE and GSM - a rotation cannot change |B|)
df["B_total"] = np.sqrt(df["Bx_gse"]**2 + df["By_gse"]**2 + df["Bz_gse"]**2)

# two useful angles
df["clock"] = np.degrees(np.arctan2(df["By_gsm"], df["Bz_gsm"]))   # 0 = northward
df["cone"]  = np.degrees(np.arccos(df["Bx_gse"] / df["B_total"]))  # 0 = sunward

print(f"\n|B| : {df['B_total'].min():.2f} - {df['B_total'].max():.2f} nT "
      f"(median {df['B_total'].median():.2f} nT)")
print(f"S/C : X = {df['x_gse'].mean()/RE:.1f} Re "
      f"({df['x_gse'].mean()/1e6:.3f} million km sunward of Earth)")

## 5. Plot Mag Data in stack

In [ ]:
FS = 14          # base font size: bump this one number to make everything bigger

df["Bt"] = np.sqrt(df["Bx_gse"]**2 + df["By_gse"]**2 + df["Bz_gse"]**2)

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

df[["Bx_gse", "By_gse", "Bz_gse"]].plot(ax=ax[0], color=["blue", "green", "red"], lw=0.8)
ax[0].set_ylabel("B GSE [nT]", fontsize=FS)

df[["Bx_gsm", "By_gsm", "Bz_gsm"]].plot(ax=ax[1], color=["blue", "green", "red"], lw=0.8)
ax[1].set_ylabel("B GSM [nT]", fontsize=FS)

df["Bt"].plot(ax=ax[2], color="black", lw=0.8)
ax[2].set_ylabel("|B| [nT]", fontsize=FS)

for a in ax:
    a.axhline(0, color="0.7", lw=0.6)
    a.tick_params(axis="both", labelsize=FS - 1)      # tick numbers
    if a.get_legend():
        a.legend(fontsize=FS - 1, ncol=3, frameon=False, loc="upper left")

ax[2].set_xlabel("Time (UTC), 10-11 October 2024", fontsize=FS)
ax[0].set_title("Aditya-L1 / MAG Level-2", fontsize=FS + 2)
plt.tight_layout()
plt.show()

## 6. Reading the ASPEX-SWIS Level-2 bulk-parameter files

**SWIS** (Solar Wind Ion Spectrometer, part of **ASPEX**) measures the solar-wind ion
distribution. The Level-2 **BLK** product contains the fitted *bulk parameters* at **5 s**
cadence:

| Variable | Meaning | Units |
|---|---|---|
| `proton_density`, `alpha_density` | number density $n_p$, $n_\alpha$ | cm⁻³ |
| `proton_bulk_speed`, `alpha_bulk_speed` | speed $|V|$ | km s⁻¹ |
| `proton_xvelocity/yvelocity/zvelocity` | velocity components in **GSE** | km s⁻¹ |
| `proton_thermal`, `alpha_thermal` | thermal speed $w$ | km s⁻¹ |
| `*_uncer` | 1-σ uncertainties | same |
| `spacecraft_[xyz]pos` | position, GSE | km |

The CDF fill value is `-1e31`, and each variable carries `VALIDMIN` / `VALIDMAX` attributes —
we use **both**, which is a habit worth teaching: a value can be finite, not equal to the fill
value, and still be outside the range the instrument team considers physical.

`cdflib` reads the CDF format; the epoch variable is `CDF_EPOCH` (milliseconds since year 0),
and `cdflib.cdfepoch.unixtime` converts it to the UNIX seconds we use everywhere.

> Unlike MAG, SWIS **does** have data gaps on these two days. That is normal and is exactly
> why the masking step is not optional.

In [ ]:
SWIS_DIR = os.path.join("Data", "ASPEX-SWISS")

# The folder also holds TH1/TH2 flux products (~25 MB each). We want only BLK,
# the bulk-moment product: density, speed, thermal speed.
swis_files = []
for d in DATES:
    hits = sorted(glob.glob(os.path.join(SWIS_DIR, f"AL1_ASW91_L2_BLK_{d}_*.cdf")))
    if not hits:
        print(f"  !! no SWIS BLK file for {d}")
        continue
    swis_files.append(hits[-1])          # newest version if several
	
assert swis_files, "no SWIS BLK files for the requested dates"
print(f"{len(swis_files)} BLK file(s), one per date:")
for f in swis_files:
    print("  ", os.path.basename(f))

cdfs = [cdflib.CDF(f) for f in swis_files]

In [ ]:
info  = cdfs[0].cdf_info()
names = info.zVariables if hasattr(info, "zVariables") else info["zVariables"]

print(f"{'variable':28s} {'shape':>14s}  units")
print("-" * 60)
for n in names:
    a   = cdfs[0].varget(n)
    att = cdfs[0].varattsget(n)
    print(f"{n:28s} {str(np.shape(a)):>14s}  {att.get('UNITS', '-')}")

In [ ]:
SWIS_VARS = ["proton_density", "proton_bulk_speed",
             "proton_xvelocity", "proton_yvelocity", "proton_zvelocity",
             "proton_thermal",
             "alpha_density", "alpha_bulk_speed", "alpha_thermal"]

def swis_frame(cdf):
    """One BLK file -> one DataFrame, indexed by UTC time."""
    epoch = cdf.varget("epoch_for_cdf_mod")
    time  = pd.to_datetime(cdflib.cdfepoch.unixtime(epoch), unit="s")
    return pd.DataFrame({v: cdf.varget(v) for v in SWIS_VARS}, index=time)

sw = pd.concat([swis_frame(c) for c in cdfs]).sort_index()

print(sw.index[0], "->", sw.index[-1], " rows:", len(sw))
sw.head()

In [ ]:
sw = sw.mask(sw <= -1e30)        # the CDF fill value is -1e31 -> NaN

print(sw.describe().T[["count", "min", "max"]].round(2))

print("\nmissing data (%):")
print((100 * sw.isna().mean()).round(1))

dt = sw.index.to_series().diff().dt.total_seconds()
print(f"\ntime step: median {dt.median():.1f} s, max {dt.max():.1f} s")
print("gaps longer than 60 s:", int((dt > 60).sum()))

In [ ]:
FS = 14

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

sw["proton_density"].plot(ax=ax[0], color="darkgreen", lw=0.8)
ax[0].set_ylabel(r"$n_p$ [cm$^{-3}$]", fontsize=FS)

sw["proton_bulk_speed"].plot(ax=ax[1], color="navy", lw=0.8)
ax[1].set_ylabel(r"$V_p$ [km s$^{-1}$]", fontsize=FS)

sw["proton_thermal"].plot(ax=ax[2], color="firebrick", lw=0.8)
ax[2].set_ylabel(r"$w_p$ [km s$^{-1}$]", fontsize=FS)

for a in ax:
    a.tick_params(axis="both", labelsize=FS - 1)

ax[2].set_xlabel("Time (UTC)", fontsize=FS)
ax[0].set_title("Aditya-L1 / ASPEX-SWIS Level-2 (BLK) - solar-wind protons", fontsize=FS + 2)
plt.tight_layout()
plt.show()

## 7. Derived plasma quantities

| Quantity | Formula | Why it matters |
|---|---|---|
| Dynamic pressure | $P_{dyn}=\rho V^2 = 1.6726\times10^{-6}\,(n_p+4n_\alpha)V^2$ nPa | compresses the magnetosphere |
| Proton temperature | $T_p = m_p w^2/2k_B$ | cold plasma is an ICME signature |
| Expected temperature | $T_{exp}=(0.031V-5.1)^2\times10^3$ K | empirical normal-solar-wind $T(V)$ (Lopez & Freeman 1986) |
| Plasma beta | $\beta = 2\mu_0 n k_B T/B^2$ | $\beta\ll1$ inside a magnetic cloud |
| Alfvén speed | $V_A = B/\sqrt{\mu_0 n m_p}$ | shock / Mach number |
| Alfvén Mach number | $M_A = V/V_A$ | shock strength |
| Merging electric field | $E_y = V_x B_z^{GSM}$ | solar-wind–magnetosphere coupling |
| Alpha-to-proton ratio | $n_\alpha/n_p$ | > 0.08 is a classic ICME/ejecta signature |

The velocity is measured by SWIS (5 s) and the field by MAG (10 s), so anything combining the
two first needs a **common time base**. We build one by averaging both onto a regular 1-minute
grid — `np.nan`-aware, so gaps stay gaps instead of being silently filled.

In [ ]:
# MAG is 10 s, SWIS is 5 s -> average both to 1 minute so they can be combined.
# .mean() ignores NaN and gives NaN for an empty minute, so gaps stay gaps.
MAG_KEEP = ["Bx_gse", "By_gse", "Bz_gse",
            "Bx_gsm", "By_gsm", "Bz_gsm",
            "B_total", "clock", "cone",
            "x_gse", "y_gse", "z_gse"]

mag_1m  = df[MAG_KEEP].resample("1min").mean()
swis_1m = sw.resample("1min").mean()          # all SWIS columns

p = mag_1m.join(swis_1m, how="outer")         # one table, one time index

print(f"{len(p)} one-minute rows, {p.index[0]} -> {p.index[-1]}")
print("  minutes with field  :", int(p['B_total'].notna().sum()))
print("  minutes with plasma :", int(p['proton_bulk_speed'].notna().sum()))
print("  minutes with both   :", int((p['B_total'].notna() & p['proton_bulk_speed'].notna()).sum()))
print("\ncolumns:", list(p.columns))

In [ ]:
n = p["proton_density"]        # cm^-3
V = p["proton_bulk_speed"]     # km/s
w = p["proton_thermal"]        # km/s
B = p["B_total"]               # nT

# proton temperature from the thermal speed:  T = m w^2 / 2k
p["Tp"] = MP * (w * 1e3)**2 / (2 * KB)                        # K

# dynamic pressure (alphas counted as 4 proton masses; missing alphas treated as 0)
p["Pdyn"] = 1.6726e-6 * (n + 4.0 * p["alpha_density"].fillna(0)) * V**2   # nPa

# plasma beta = thermal pressure / magnetic pressure
p["beta"] = 2 * MU0 * (n * 1e6) * KB * p["Tp"] / (B * 1e-9)**2

# Alfven speed and Mach number
p["Va"] = (B * 1e-9) / np.sqrt(MU0 * (n * 1e6) * MP) / 1e3    # km/s
p["Ma"] = V / p["Va"]

# merging electric field Ey = |Vx| * Bz  -> POSITIVE when the field is southward
p["Ey"] = -p["proton_xvelocity"].abs() * p["Bz_gsm"] * 1e-3    # mV/m

# alpha-to-proton ratio: > 0.08 is a classic ICME signature
p["alpha_ratio"] = p["alpha_density"] / n

DERIVED = ["Tp", "Pdyn", "beta", "Va", "Ma", "Ey", "alpha_ratio"]
print(p[DERIVED].describe().T[["count", "min", "mean", "max"]].round(3))

## 8. The combined MAG + SWIS overview

Seven panels on one shared time axis, built with plain `matplotlib`. Keep the label next to the data it describes — that habit is worth more than any plotting library.

In [ ]:
FS = 13

fig, ax = plt.subplots(7, 1, figsize=(12, 15), sharex=True)

# 1 - field magnitude
p["B_total"].plot(ax=ax[0], color="black", lw=1.5)
ax[0].set_ylabel("|B|\n[nT]", fontsize=FS)

# 2 - field components in GSE
p[["Bx_gse", "By_gse", "Bz_gse"]].plot(ax=ax[1], color=["blue", "green", "red"], lw=1.5)
ax[1].set_ylabel("B GSE\n[nT]", fontsize=FS)
ax[1].axhline(0, color="0.7", lw=1.5)
ax[1].legend(["Bx", "By", "Bz"], ncol=3, fontsize=FS - 2, frameon=False, loc="upper left")

# 3 - proton density
p["proton_density"].plot(ax=ax[2], color="darkgreen", lw=1.5)
ax[2].set_ylabel(r"$n_p$" "\n" r"[cm$^{-3}$]", fontsize=FS)

# 4 - bulk speed
p["proton_bulk_speed"].plot(ax=ax[3], color="navy", lw=1.5)
ax[3].set_ylabel(r"$V_p$" "\n" r"[km s$^{-1}$]", fontsize=FS)

# 5 - proton temperature (log, it spans two decades)
(p["Tp"] / 1e3).plot(ax=ax[4], color="firebrick", lw=1.5, logy=True)
ax[4].set_ylabel(r"$T_p$" "\n[Kelvin]", fontsize=FS)

# 6 - dynamic pressure
p["Pdyn"].plot(ax=ax[5], color="darkorange", lw=1.5)
ax[5].set_ylabel(r"$P_{dyn}$" "\n[nPa]", fontsize=FS)

# 7 - plasma beta (log; 0.1 marks the magnetic-cloud threshold)
p["beta"].plot(ax=ax[6], color="teal", lw=1.0, logy=True)
ax[6].axhline(1, color="0.6", lw=1.5, ls=":")
ax[6].set_ylabel(r"$\beta$", fontsize=FS)

for a in ax:
    a.tick_params(axis="both", labelsize=FS - 2)

ax[6].set_xlabel("Time (UTC)", fontsize=FS)
ax[0].set_title("Aditya-L1 in situ  |  MAG + ASPEX-SWIS  |  1-minute averages",
                fontsize=FS + 3)
plt.tight_layout()
# fig.savefig("AL1_overview.png", dpi=150, bbox_inches="tight")
plt.show()

### What the overview shows

Read the panels together, not one at a time — that is the whole skill:

* **Quiet solar wind** until ~15 UT on 10 October: $|B|\approx5$ nT, $V\approx400$ km s⁻¹,
  $n_p\approx4$ cm⁻³, $\beta\sim1$.
* **An abrupt, simultaneous jump** in $|B|$, $n_p$, $V$ and $T_p$ — the fingerprint of a
  **fast forward shock**. Everything jumps at once and in the same sense; that simultaneity is
  what distinguishes a shock from a mere discontinuity.
* **A turbulent, compressed, hot interval** behind it — the **sheath**: high $|B|$ but wildly
  fluctuating direction, high density, $\beta\gtrsim1$.
* **A smooth, strong-field, cold, low-$\beta$ interval** afterwards with the field direction
  rotating slowly and monotonically — the **magnetic cloud / ICME ejecta**.
* The speed stays high and decays slowly through 11 October — the expansion signature of the
  ejecta.

## 9. ASPEX-STEPS — the energetic particles

**STEPS** (Supra-Thermal and Energetic Particle Spectrometer, the second half of ASPEX) measures
ions from ~0.25 to >20 MeV — a thousand times more energetic than the solar wind SWIS sees. It
has **six look directions**, which is the point of the instrument: energetic particles stream
*along* the magnetic field, so the difference between detectors is a measurement of anisotropy.

| L2 file | detector | looks towards |
|---|---|---|
| `..._PS_...` | **Parker Spiral**, inner + outer | along the nominal Parker spiral field direction |
| `..._EP_...` | **Earth Pointing**, inner + outer | anti-sunward, towards Earth |
| `..._IM_...` | **Intermediate** | between PS and EP |
| `..._NP_...` | **North Pointing** | ecliptic north |

Each file holds differential directional flux in **11 energy bins** (0.25/0.3 – 6 MeV) at a
**10-minute** cadence, in units of counts s⁻¹ MeV⁻¹ sr⁻¹ cm⁻². The energy bins come as an
`(11, 2)` array of `[lower, upper]` edges — we turn that into the 12 edges a `pcolormesh` wants.

Note the cadence: **10 minutes**, against MAG's 10 seconds. This is why `stack_plot` lets each
panel carry its own time array.

### A note on units and on zeros

The flux is **differential directional flux**: counts per second, per MeV of energy width, per
steradian of solid angle, per cm² of detector area. Dividing by the energy width is what makes
channels of different widths comparable — always check whether a flux is differential or
integral before you compare two instruments.

Zeros are set to NaN above, deliberately: a zero-count bin is not "zero flux", it is "no counts
in this accumulation", and on a logarithmic colour scale zero has no place. Distinguishing
*measured zero* from *no measurement* is one of the recurring judgement calls in particle data.

## 10. The STEPS spectrogram and the anisotropy

In [ ]:
STEPS_DIR = os.path.join("Data", "ASPEX-STEPS")

# STEPS writes one L2 file per look direction per day; the tag in the file name
# is the detector.
STEPS_TAGS = {
    "PS": "Parker Spiral",
    "EP": "Earth Pointing",
    "IM": "Intermediate",
    "NP": "North Pointing",
}

# --- what is on disk? ----------------------------------------------------------
print("L2 files found:")
for f in sorted(glob.glob(os.path.join(STEPS_DIR, "AL1_AST91_L2_*.cdf"))):
    print("  ", os.path.basename(f))

# --- pick the files for the dates chosen in section 2 --------------------------
steps_files = {}
for tag in STEPS_TAGS:
    hits = []
    for d in DATES:
        hits += sorted(glob.glob(os.path.join(STEPS_DIR, f"AL1_AST91_L2_{tag}_{d}_*.cdf")))
    if not hits:
        print(f"  !! no STEPS {tag} file for {DATES}")
    else:
        steps_files[tag] = hits

assert steps_files, "no STEPS files for the requested dates"

steps_cdfs = {tag: [cdflib.CDF(f) for f in fs] for tag, fs in steps_files.items()}

print("\ndetectors loaded:")
for tag in steps_cdfs:
    print(f"   {tag} = {STEPS_TAGS[tag]:16s} ({len(steps_cdfs[tag])} file(s))")

In [ ]:
cdf   = steps_cdfs["PS"][0]
info  = cdf.cdf_info()
names = info.zVariables if hasattr(info, "zVariables") else info["zVariables"]

print(f"{'variable':30s} {'shape':>14s}  units")
print("-" * 62)
for n in names:
    a   = cdf.varget(n)
    att = cdf.varattsget(n)
    print(f"{n:30s} {str(np.shape(a)):>14s}  {att.get('UNITS', '-')}")

In [ ]:
# flux variable and energy-bin variable inside each file
STEPS_VARS = {
    "PS": ("PS_Inner_allspecies", "Ebin_inner_allspecies"),
    "EP": ("EP_Inner_allspecies", "Ebin_inner_allspecies"),
    "IM": ("IM_allspecies",       "Ebin_allspecies"),
    "NP": ("NP_allspecies",       "Ebin_allspecies"),
}

# --- energy bins: the file gives (11, 2) = [lower, upper] edges in MeV ---------
ebin = steps_cdfs["PS"][0].varget(STEPS_VARS["PS"][1])
E_LO, E_HI = ebin[:, 0], ebin[:, 1]
E_CTR   = np.sqrt(E_LO * E_HI)          # geometric centre of each channel
E_EDGES = np.append(E_LO, E_HI[-1])     # 12 monotonic edges -> what pcolormesh wants
E_COLS  = [f"{lo:.2f}-{hi:.2f}" for lo, hi in zip(E_LO, E_HI)]

print(f"channels: {len(E_LO)}    range: {E_LO[0]:.2f} - {E_HI[-1]:.2f} MeV")
for i, (lo, hi) in enumerate(zip(E_LO, E_HI)):
    print(f"   ch {i:2d}: {lo:6.2f} - {hi:6.2f} MeV   (width {hi - lo:5.2f})")

# --- one DataFrame per look direction: rows = time, columns = energy channel ---
def steps_frame(cdfs, fluxvar):
    """The files of one look direction -> one DataFrame, indexed by UTC time."""
    frames = []
    for c in cdfs:
        time = pd.to_datetime(cdflib.cdfepoch.unixtime(c.varget("Epoch")), unit="s")
        frames.append(pd.DataFrame(c.varget(fluxvar), index=time, columns=E_COLS))
    return pd.concat(frames).sort_index()

st = {tag: steps_frame(steps_cdfs[tag], STEPS_VARS[tag][0]) for tag in steps_cdfs}

ps = st["PS"]                                    # the one we look at most
dt = ps.index.to_series().diff().dt.total_seconds()
print(f"\n{ps.index[0]} -> {ps.index[-1]}   rows: {len(ps)}")
print(f"cadence: median {dt.median()/60:.1f} min, max step {dt.max()/60:.1f} min")
ps.head()

In [ ]:
# Three kinds of unusable sample:
#   the CDF fill value (-1e31), non-finite values, and exact zeros.
# A zero-count bin is "no counts in this accumulation", not "zero flux", and a
# logarithmic colour scale has no place for it -> NaN.
for tag in st:
    d = st[tag]
    st[tag] = d.mask(~np.isfinite(d) | (d <= -1e29) | (d <= 0))

print(f"{'detector':16s} {'rows':>6s} {'bad %':>7s} {'min flux':>12s} {'max flux':>12s}")
print("-" * 58)
for tag, d in st.items():
    print(f"{STEPS_TAGS[tag]:16s} {len(d):6d} {100 * d.isna().mean().mean():6.1f}% "
          f"{d.min().min():12.3g} {d.max().max():12.3g}")

# colour limits for the spectrograms below (percentiles, so one hot bin
# does not flatten the whole plot)
allflux    = np.concatenate([d.to_numpy().ravel() for d in st.values()])
VMIN, VMAX = np.nanpercentile(allflux, [1, 99.9])
print(f"\ncolour scale to be used: {VMIN:.3g} - {VMAX:.3g}")

# per-channel detail for one detector
print(f"\n{STEPS_TAGS['PS']}, per energy channel:")
print(ps.describe().T[["count", "min", "max"]].round(3))

In [ ]:
FS = 14

def time_edges(index):
    """Cell centres -> cell edges for pcolormesh, as matplotlib date numbers.

    np.append on a DatetimeIndex gives an object array of Timestamps, which
    pcolormesh cannot use; date2num turns it into the floats matplotlib
    actually plots dates as.
    """
    t    = index.to_numpy()                 # datetime64[ns]
    step = np.median(np.diff(t))            # the cadence, as timedelta64
    return mdates.date2num(np.append(t, t[-1] + step))


fig, axes = plt.subplots(len(st), 2, figsize=(13, 3 * len(st)), sharex="col",
                         gridspec_kw={"width_ratios": [40, 1],
                                      "hspace": 0.12, "wspace": 0.02})
ax, cax = axes[:, 0], axes[:, 1]

for a, c, (tag, d) in zip(ax, cax, st.items()):
    # pcolormesh wants one more edge than data point, in time as well as energy
    m = a.pcolormesh(time_edges(d.index), E_EDGES, d.to_numpy().T,
                     norm=LogNorm(vmin=VMIN, vmax=VMAX), cmap="jet")
    a.set_yscale("log")
    a.xaxis_date()                          # tell the axis those floats are dates
    a.set_ylabel(f"{tag}\n[MeV]", fontsize=FS)
    a.tick_params(axis="both", labelsize=FS - 2)
    fig.colorbar(m, cax=c)
    c.tick_params(labelsize=FS - 3)

cax[0].set_title(r"cts s$^{-1}$MeV$^{-1}$" "\n" r"sr$^{-1}$cm$^{-2}$", fontsize=FS - 4)

ax[-1].xaxis.set_major_locator(mdates.HourLocator(byhour=range(0, 24, 6)))
ax[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%H:%M"))
ax[-1].set_xlabel("Time (UTC)", fontsize=FS)
ax[0].set_title("Aditya-L1 / ASPEX-STEPS Level-2 - energetic ions, four look directions",
                fontsize=FS + 2)
fig.savefig("AL1_STEPS_spectrograms.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
FS = 14
N_CH      = 3                      # lowest three channels, 0.25 - 0.68 MeV
QUIET_END = "2024-10-10 12:00"     # everything before this is the pre-event level

# one low-energy flux per look direction, in one table
low = pd.DataFrame({STEPS_TAGS[tag]: d.iloc[:, :N_CH].mean(axis=1)
                    for tag, d in st.items()})

quiet = low.index < pd.Timestamp(QUIET_END)
print(f"{'detector':16s} {'quiet':>11s} {'peak':>11s} {'factor':>8s}   peak at")
print("-" * 68)
for name in low:
    f = low[name]
    base = f[quiet].median()
    print(f"{name:16s} {base:11.3g} {f.max():11.3g} {f.max()/base:8.1f}   "
          f"{f.idxmax():%Y-%m-%d %H:%M}")

fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True,
                       gridspec_kw={"height_ratios": [2, 1], "hspace": 0.08})

low.plot(ax=ax[0], lw=1.2, logy=True,
         color=["crimson", "navy", "darkgreen", "darkorange"])
ax[0].set_ylabel(f"{E_LO[0]:.2f}-{E_HI[N_CH-1]:.2f} MeV\nflux", fontsize=FS)
ax[0].legend(fontsize=FS - 2, ncol=2, frameon=False, loc="upper left")

# anisotropy in one number: strongest look direction / weakest
(low.max(axis=1) / low.min(axis=1)).plot(ax=ax[1], color="black", lw=1.0, logy=True)
ax[1].axhline(1, color="0.6", lw=0.8, ls=":")
ax[1].set_ylabel("max / min\n(anisotropy)", fontsize=FS)

for a in ax:
    a.tick_params(axis="both", labelsize=FS - 1)

ax[1].set_xlabel("Time (UTC)", fontsize=FS)
ax[0].set_title("ASPEX-STEPS - low-energy flux in four look directions", fontsize=FS + 2)
fig.savefig("AL1_STEPS_anisotropy.png", dpi=150, bbox_inches="tight")
plt.show()

### What STEPS is showing

* The flux starts rising **hours before** the shock arrives — these are particles accelerated at
  the shock front and streaming ahead of it along the magnetic field. This is the classic
  **energetic storm particle (ESP)** signature and, in operational space weather, it is the
  *earliest warning* you get: the particles outrun their own shock.
* The flux **peaks right at the shock**, then falls.
* Afterwards, through 11 October, the flux settles **below** its pre-event level. That is a
  **Forbush decrease**: the strong, closed magnetic field of the ICME sweeps galactic cosmic
  rays out of the volume it occupies, so the background drops for a day or more.
* Look closely at *when* that drop happens — it is a sharp step late on 10 October, not a
  gradual decay. Project 1 will place the **leading edge of the magnetic cloud** at almost
  exactly that time using entirely different data (field smoothness, $\beta$, density). An
  energetic-particle boundary confirming a magnetic-field boundary is a genuinely independent
  check, and the sort of cross-instrument argument that makes a result publishable.
* The four look directions differ by a large factor during the rise — the particle distribution
  is **anisotropic**, beamed along the field. Once the shock has passed, the directions converge,
  because scattering has isotropised the population.

Together with the MAG and SWIS panels, these are three independent instruments telling one
consistent story about a single structure passing the spacecraft. Making them agree — and
noticing when they do not — is what in-situ space physics is.

---
## 11. The grand overview — three payloads, one event

This is the figure to put on the screen at the start of the session. Every panel comes from a
different measurement, at a different cadence, and they all describe the same structure passing
Aditya-L1 over two days.

Read it top to bottom as a single sentence: *a shock arrives, compressing and heating the
plasma; energetic particles announce it in advance; behind it comes a turbulent sheath and then
a strong, smooth, cold magnetic cloud with a long interval of southward field.*

In [ ]:
FS = 13

fig, axes = plt.subplots(9, 2, figsize=(13, 20), sharex="col",
                         gridspec_kw={"width_ratios": [40, 1],
                                      "hspace": 0.08, "wspace": 0.02})
ax, cax = axes[:, 0], axes[:, 1]
for c in cax[1:]:
    c.axis("off")                    # only the spectrogram needs a colour bar

# 1 - STEPS spectrogram (10 min cadence, its own time axis)
d = st["PS"]
m = ax[0].pcolormesh(time_edges(d.index), E_EDGES, d.to_numpy().T,
                     norm=LogNorm(vmin=VMIN, vmax=VMAX), cmap="viridis")
ax[0].set_yscale("log")
ax[0].xaxis_date()
ax[0].set_ylabel("STEPS PS\n[MeV]", fontsize=FS)
fig.colorbar(m, cax=cax[0])
cax[0].tick_params(labelsize=FS - 4)

# 2 - field components, GSM            (everything below is the 1-minute table p)
for col, colour, lbl in zip(["Bx_gsm", "By_gsm", "Bz_gsm"], BXYZ, ["Bx", "By", "Bz"]):
    ax[1].plot(p.index, p[col], color=colour, lw=1.0, label=lbl)
ax[1].axhline(0, color="0.7", lw=0.6)
ax[1].set_ylabel("B GSM\n[nT]", fontsize=FS)
ax[1].legend(ncol=3, fontsize=FS - 3, frameon=False, loc="upper left")

# 3 to 9 - one line each: (column, colour, label, log?)
PANELS = [
    ("B_total",           "black",       "|B|\n[nT]",                          False),
    ("proton_density",    "darkgreen",   r"$n_p$" "\n" r"[cm$^{-3}$]",         False),
    ("proton_bulk_speed", "navy",        r"$V_p$" "\n" r"[km s$^{-1}$]",       False),
    ("Tp",                "firebrick",   r"$T_p$" "\n[K]",                     True),
    ("Pdyn",              "darkorange",  r"$P_{dyn}$" "\n[nPa]",               False),
    ("beta",              "teal",        r"$\beta$",                           True),
    ("Ey",                "darkviolet",  r"$E_y$" "\n" r"[mV m$^{-1}$]",       False),
]

for a, (col, colour, lbl, logy) in zip(ax[2:], PANELS):
    a.plot(p.index, p[col], color=colour, lw=1.0)
    if logy:
        a.set_yscale("log")
    a.set_ylabel(lbl, fontsize=FS)

ax[7].axhline(0.1, color="0.6", lw=0.8, ls=":")     # magnetic-cloud threshold
ax[8].axhline(0.0, color="0.7", lw=0.6)

for a in ax:
    a.tick_params(axis="both", labelsize=FS - 3)

ax[-1].xaxis.set_major_locator(mdates.HourLocator(byhour=range(0, 24, 6)))
ax[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%H:%M"))
ax[-1].set_xlabel("Time (UTC)", fontsize=FS)
ax[0].set_title("Aditya-L1 in situ  |  ASPEX-STEPS + MAG + ASPEX-SWIS", fontsize=FS + 3)

# fig.savefig("AL1_grand_overview.png", dpi=150, bbox_inches="tight")
plt.show()

### Reading the grand overview

| Panel | Quiet solar wind | After the shock |
|---|---|---|
| STEPS flux | steady background | rises *before* the shock, peaks at it, then a Forbush decrease |
| $B$, $\vert B\vert$ | ~5 nT, slowly varying | 30–47 nT; turbulent, then smooth |
| $n_p$ | ~4 cm⁻³ | ~20 cm⁻³ in the sheath, low in the cloud |
| $V_p$ | ~410 km s⁻¹ | jumps to ~830 km s⁻¹, decays slowly |
| $T_p$ | ~150 kK | shock-heated by a factor of several |
| $P_{dyn}$ | ~1 nPa | up to ~20 nPa — the magnetosphere is squeezed |
| $\beta$ | ~0.4 | >1 in the sheath, ≪0.1 in the cloud |
| $E_y$ | ~0 | sustained ~10–20 mV m⁻¹ — the storm driver |

**The two projects start here.**

* **Project 1** (`Project1_CME_Substructures.ipynb`) asks *where exactly* the shock, the sheath
  and the magnetic cloud begin and end, and what the field geometry inside the cloud is.
* **Project 2** (`Project2_Geoeffectiveness.ipynb`) asks *what happened at Earth* — propagating
  these measurements to the bow shock and comparing them with SYM-H and AE.

Both begin from exactly the code you have just run.
